# 02 — Silver: Cleaning & Transforms

**Tickets:** I-03, I-04, I-05  
**Purpose:** Clean Bronze data into a reliable Silver table — cast types, drop corrupt rows, handle missing values, add derived columns.

---

## Setup

In [0]:
from src.constants import BRONZE_TABLE
from src.transforms import (
    # I-03
    add_zone_bins,
    cast_datetime_columns,
    cast_numeric_columns,
    clean_gps_coordinates,
    deduplicate,
    drop_corrupt_rows,
    recover_rate_code_id,
    standardise_column_names,
    # I-04
    cap_monetary_outliers,
    drop_duration_anomalies,
    drop_extreme_outliers,
    drop_invalid_fares,
    drop_zero_distance_trips,
    drop_zero_passenger_trips,
)

print("Setup complete")

## Read Bronze table

In [0]:
bronze_df = spark.read.table(BRONZE_TABLE)
print(f"Bronze rows: {bronze_df.count():,}")
print(f"Bronze columns: {bronze_df.columns}")
bronze_df.printSchema()

## I-03 — Column name standardisation & type casting

In [0]:
bronze_count = bronze_df.count()

# I-03a — Standardise column names (VendorID -> vendor_id, RateCodeID -> rate_code_id)
silver_df = standardise_column_names(bronze_df)

# I-03b — Recover rate_code_id from _rescued_data JSON (Finding #1: 73% of rows affected)
silver_df = recover_rate_code_id(silver_df)

# I-03c — Cast datetime columns to TimestampType
silver_df = cast_datetime_columns(silver_df)

# I-03d — Cast numeric columns to correct types (int / double)
silver_df = cast_numeric_columns(silver_df)

# I-03e — Deduplicate exact duplicate rows (Finding #9: 772 rows)
silver_df = deduplicate(silver_df)

# I-03f — Drop structurally corrupt rows (nulls in required fields)
silver_df = drop_corrupt_rows(silver_df)

# I-03g — Clean GPS coordinates: NULL out (0,0) and out-of-NYC-bbox (Finding #10)
silver_df = clean_gps_coordinates(silver_df)

# I-03h — Add grid-binned zone columns from cleaned lat/lon (needed by Gold: BQ-1, BQ-2, A-02, A-03)
silver_df = add_zone_bins(silver_df)

after_count = silver_df.count()
print(f"Bronze rows:  {bronze_count:,}")
print(f"Silver rows:  {after_count:,}")
print(
    f"Dropped:      {bronze_count - after_count:,} ({(bronze_count - after_count) / bronze_count * 100:.2f}%)"
)
silver_df.printSchema()
display(silver_df.limit(5))

## I-04 — Handle missing values & outliers

### Decisions & rationale

Each decision maps to a finding from [00_explore](#notebook-4211759161263644). Thresholds live in `src/constants.py`; transform functions in `src/transforms.py`.

| Decision | Finding | Action | Rationale | Rows affected |
|----------|---------|--------|-----------|---------------|
| A | #6 | **Drop** `passenger_count = 0` | Only 0.02% of data; structurally invalid for a completed trip | 16,428 |
| B | #8 | **Drop** `trip_distance > 100 mi` | Likely GPS errors or inter-city rides outside normal taxi service | 914 |
| C | #8, #12 | **Drop** `fare > $500`, `total > $1,000` | Extreme outliers (max $3.95M) are data entry errors | ~668 |
| D | #12 | **Set negative `tip_amount` to $0** | Row is otherwise valid; negative tip is likely a recording error | 840 |
| E | #12 | **Cap `tip_amount` at $200** | Values up to $3.95M are clearly erroneous; $200 is a generous upper bound | ~1,368 |
| F | #12 | **NULL out `extra` \~\~ {0, 0.5, 1.0}** | Standard surcharges are $0.50 (rush hour) / $1.00 (overnight); irregular values are data errors | ~168K |
| G | #11 | **Drop** negative/zero duration and **> 24 h** | Negative/zero are impossible; > 24 h (max \~381 days) are errors. Trips ≤ 24 h retained (includes legitimate JFK/Newark airport rides) | ~102K |
| H | #7 | **No action** — defer to ML stage | 1,150 rows with tip > 0 on non-card payments; kept in Silver, tip models (BQ-4) should train on `payment_type = 1` only | 0 (kept) |

**Remaining nulls:** After I-03's `drop_corrupt_rows` removes rows with null timestamps/fare/total/payment_type, the only nullable columns are `rate_code_id` (code 99 → NULL), GPS coordinates (NULLed by `clean_gps_coordinates`), `extra` (non-standard → NULL above), and `store_and_fwd_flag`. These are acceptable — downstream consumers handle them appropriately.

In [0]:
before_i04 = silver_df.count()

# I-04a — Drop zero-distance trips (Finding #3: 564K rows, 0.60%)
silver_df = drop_zero_distance_trips(silver_df)

# I-04b — Drop non-positive fares & negative totals (Findings #4, #5)
silver_df = drop_invalid_fares(silver_df)

# I-04c — Drop zero-passenger trips (Finding #6: 16K rows, 0.02%)
silver_df = drop_zero_passenger_trips(silver_df)

# I-04d — Drop extreme distance/fare/total outliers (Findings #8, #12)
silver_df = drop_extreme_outliers(silver_df)

# I-04e — Drop impossible trip durations: negative, zero, or > 24 h (Finding #11)
silver_df = drop_duration_anomalies(silver_df)

# I-04f — Cap/correct monetary outliers: negative tips → 0, tip cap $200,
#          extra surcharge restricted to {0, 0.5, 1.0} (Finding #12)
silver_df = cap_monetary_outliers(silver_df)

after_i04 = silver_df.count()
print(f"Before I-04:  {before_i04:,}")
print(f"After I-04:   {after_i04:,}")
print(
    f"I-04 dropped: {before_i04 - after_i04:,} ({(before_i04 - after_i04) / before_i04 * 100:.2f}%)"
)
display(silver_df.limit(5))

## I-05 — Derived columns

In [0]:
# TODO: add trip_duration_min, hour_of_day, day_of_week, is_weekend

## Write Silver Delta table

In [0]:
# TODO: write to Silver Delta table